In [ ]:
import os 
from dotenv import load_dotenv
load_dotenv()
os.environ["NEO4J_URI"]=os.getenv("NEO4J_URI")
os.environ["NEO4J_USERNAME"]=os.getenv("NEO4J_USERNAME")
os.environ["NEO4J_PASSWORD"]=os.getenv("NEO4J_PASSWORD")

In [ ]:
# Neo4j credentials are loaded from .env file (see cell above using load_dotenv)
# Copy .env.example to .env and fill in your actual credentials
# NEO4J_URI, NEO4J_USERNAME, NEO4J_PASSWORD are set via os.getenv() above

NEO4J_URI = os.getenv("NEO4J_URI")
NEO4J_USERNAME = os.getenv("NEO4J_USERNAME")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")

In [ ]:
import os
os.environ["NEO4J_URI"] = NEO4J_URI
os.environ["NEO4J_USERNAME"] = NEO4J_USERNAME
os.environ["NEO4J_PASSWORD"] = NEO4J_PASSWORD

In [ ]:
print(NEO4J_URI, NEO4J_USERNAME)

In [ ]:
from langchain_community.graphs import Neo4jGraph
graph=Neo4jGraph(url=NEO4J_URI,username=NEO4J_USERNAME,password=NEO4J_PASSWORD)

In [ ]:
movies_query=""" 
LOAD CSV WITH HEADERS FROM
'https://raw.githubusercontent.com/tomasonjo/blog-datasets/main/movies/movies_small.csv' as row

MERGE(m:Movie{id:row.movieId})
SET m.released = date(row.released),
    m.title = row.title,
    m.imdbRating = toFloat(row.imdbRating)
FOREACH (director in split(row.director, '|') | 
    MERGE (p:Person {name:trim(director)})
    MERGE (p)-[:DIRECTED]->(m))
FOREACH (actor in split(row.actors, '|') | 
    MERGE (p:Person {name:trim(actor)})
    MERGE (p)-[:ACTED_IN]->(m))
FOREACH (genre in split(row.genres, '|') | 
    MERGE (g:Genre {name:trim(genre)})
    MERGE (m)-[:IN_GENRE]->(g))

"""

In [ ]:
movies_query

In [ ]:
graph.query(movies_query)

In [ ]:
graph.refresh_schema()
print(graph.schema)

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

groq_api_key=os.getenv("GROQ_API_KEY")

In [ ]:
from langchain_groq import ChatGroq

llm=ChatGroq(groq_api_key=groq_api_key,model='llama-3.1-8b-instant')
llm

In [ ]:
from langchain.chains import GraphCypherQAChain
chain=GraphCypherQAChain.from_llm(graph=graph,llm=llm,verbose=True)
chain

In [ ]:
response=chain.invoke({"query":"Who was the director of the movie Casino"})
response

### Prompting Strategy

In [ ]:
from langchain.chains import GraphCypherQAChain
chain=GraphCypherQAChain.from_llm(graph=graph,llm=llm,exclude_type=["Genre"],verbose=True)
chain

In [ ]:
examples = [
    {
        "question": "How many artists are there?",
        "query": "MATCH (a:Person)-[:ACTED_IN]->(:Movie) RETURN count(DISTINCT a)",
    },
    {
        "question": "Which actors played in the movie Casino?",
        "query": "MATCH (m:Movie {{title: 'Casino'}})<-[:ACTED_IN]-(a) RETURN a.name",
    },
    {
        "question": "How many movies has Tom Hanks acted in?",
        "query": "MATCH (a:Person {{name: 'Tom Hanks'}})-[:ACTED_IN]->(m:Movie) RETURN count(m)",
    },
    {
        "question": "Which actors have worked in movies from both the comedy and action genres?",
        "query": "MATCH (a:Person)-[:ACTED_IN]->(:Movie)-[:IN_GENRE]->(g1:Genre), (a)-[:ACTED_IN]->(:Movie)-[:IN_GENRE]->(g2:Genre) WHERE g1.name = 'Comedy' AND g2.name = 'Action' RETURN DISTINCT a.name",
    },
    {
        "question": "Which directors have made movies with at least three different actors named 'John'?",
        "query": "MATCH (d:Person)-[:DIRECTED]->(m:Movie)<-[:ACTED_IN]-(a:Person) WHERE a.name STARTS WITH 'John' WITH d, COUNT(DISTINCT a) AS JohnsCount WHERE JohnsCount >= 3 RETURN d.name",
    },
    {
        "question": "Identify movies where directors also played a role in the film.",
        "query": "MATCH (p:Person)-[:DIRECTED]->(m:Movie), (p)-[:ACTED_IN]->(m) RETURN m.title, p.name",
    },
    {
        "question": "Find the actor with the highest number of movies in the database.",
        "query": "MATCH (a:Person)-[:ACTED_IN]->(m:Movie) RETURN a.name, COUNT(m) AS movieCount ORDER BY movieCount DESC LIMIT 1",
    },
]

In [50]:
from langchain.prompts import FewShotPromptTemplate, PromptTemplate

example_prompt = PromptTemplate(
    input_variables=["question", "query"],
    template="Question: {question}\nCypher: {query}\n"
)

prompt = FewShotPromptTemplate(
    examples=examples[:5],
    example_prompt=example_prompt,
    prefix=(
        "You are a Neo4j Cypher expert.\n"
        "Given the schema and examples, write a Cypher query that answers the question.\n"
        "IMPORTANT RULES:\n"
        "- OUTPUT ONLY THE CYPHER QUERY.\n"
        "- Do NOT include explanations, quotes, markdown, or the word 'Cypher:'.\n"
        "- Start the output with MATCH, WITH, CALL, or RETURN.\n\n"
        "Schema:\n{schema}\n\n"
        "Examples:\n"
    ),
    suffix=(
        "Question: {question}\n"
        "Cypher:"
    ),
    input_variables=["question", "schema"],
)


In [51]:
from langchain.chains import GraphCypherQAChain
chain=GraphCypherQAChain.from_llm(graph=graph,llm=llm,cypher_prompt=prompt,verbose=True)
chain

GraphCypherQAChain(verbose=True, graph=<langchain_community.graphs.neo4j_graph.Neo4jGraph object at 0x0000025457680AD0>, cypher_generation_chain=LLMChain(prompt=FewShotPromptTemplate(input_variables=['question', 'schema'], examples=[{'question': 'How many artists are there?', 'query': 'MATCH (a:Person)-[:ACTED_IN]->(:Movie) RETURN count(DISTINCT a)'}, {'question': 'Which actors played in the movie Casino?', 'query': "MATCH (m:Movie {{title: 'Casino'}})<-[:ACTED_IN]-(a) RETURN a.name"}, {'question': 'How many movies has Tom Hanks acted in?', 'query': "MATCH (a:Person {{name: 'Tom Hanks'}})-[:ACTED_IN]->(m:Movie) RETURN count(m)"}, {'question': 'Which actors have worked in movies from both the comedy and action genres?', 'query': "MATCH (a:Person)-[:ACTED_IN]->(:Movie)-[:IN_GENRE]->(g1:Genre), (a)-[:ACTED_IN]->(:Movie)-[:IN_GENRE]->(g2:Genre) WHERE g1.name = 'Comedy' AND g2.name = 'Action' RETURN DISTINCT a.name"}, {'question': "Which directors have made movies with at least three differ

In [52]:
schema_text = graph.schema
prompt.format(
    question="How many actors are there?",
    schema=schema_text
)

"You are a Neo4j Cypher expert.\nGiven the schema and examples, write a Cypher query that answers the question.\nIMPORTANT RULES:\n- OUTPUT ONLY THE CYPHER QUERY.\n- Do NOT include explanations, quotes, markdown, or the word 'Cypher:'.\n- Start the output with MATCH, WITH, CALL, or RETURN.\n\nSchema:\nNode properties:\nPerson {name: STRING}\nMovie {title: STRING, released: DATE, id: STRING, imdbRating: FLOAT}\nGenre {name: STRING}\nRelationship properties:\n\nThe relationships:\n(:Person)-[:DIRECTED]->(:Movie)\n(:Person)-[:ACTED_IN]->(:Movie)\n(:Movie)-[:IN_GENRE]->(:Genre)\n\nExamples:\n\n\nQuestion: How many artists are there?\nCypher: MATCH (a:Person)-[:ACTED_IN]->(:Movie) RETURN count(DISTINCT a)\n\n\nQuestion: Which actors played in the movie Casino?\nCypher: MATCH (m:Movie {title: 'Casino'})<-[:ACTED_IN]-(a) RETURN a.name\n\n\nQuestion: How many movies has Tom Hanks acted in?\nCypher: MATCH (a:Person {name: 'Tom Hanks'})-[:ACTED_IN]->(m:Movie) RETURN count(m)\n\n\nQuestion: Which

In [53]:
chain.invoke("Which actors played in the movie Casino?")

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


Generated Cypher:
MATCH (m:Movie {title: 'Casino'})<-[:ACTED_IN]-(a) RETURN a.name
Full Context:
[{'a.name': 'Robert De Niro'}, {'a.name': 'Joe Pesci'}, {'a.name': 'Sharon Stone'}, {'a.name': 'James Woods'}]

> Finished chain.


{'query': 'Which actors played in the movie Casino?',
 'result': 'Robert De Niro, Joe Pesci, Sharon Stone, James Woods played in the movie Casino.'}